In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

pd.set_option('display.max_columns', 60); pd.set_option('display.width', 220)
mpl.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.22,
                      'axes.titlesize': 11, 'axes.labelsize': 10, 'legend.fontsize': 9})

_CANDIDATES = ["../../data/processed/windows.parquet",
               "data/processed/windows.parquet",
               "windows.parquet",
               "/mnt/user-data/uploads/windows.parquet"]
_src = next((p for p in _CANDIDATES if Path(p).exists()), None)
if _src is None:
    raise FileNotFoundError(f"no windows.parquet found; tried {_CANDIDATES}")
w = pd.read_parquet(_src)
print(f"Loaded {_src}")
print(f"{w.shape[0]:,} windows x {w.shape[1]:,} columns")

PARTICIPANTS = sorted(w['participantId'].unique())
_PALETTE = plt.get_cmap('tab20').colors
_PCOLOR = {p: _PALETTE[i % len(_PALETTE)] for i, p in enumerate(PARTICIPANTS)}
def pcolor(p):
    return _PCOLOR.get(p, '0.5')

METADATA_COLS = [c for c in w.columns if c.endswith(('_afforded', '_observed', '_straddle_conflict'))] + \
    ['sessionId', 'participantId', 'deviceFamily', 'window_index', 'window_start_s', 'window_end_s',
     'taskIndex', 'taskType', 'activeArea', 'task_pass', 'task_start_s', 'task_end_s', 'task_boundary_straddle']
FAMILIES = {
    'typing':   [c for c in w.columns if c.startswith('typing_') and c not in METADATA_COLS],
    'tap':      [c for c in w.columns if c.startswith('tap_') and c not in METADATA_COLS],
    'gesture':  [c for c in w.columns if c.startswith('gesture_') and c not in METADATA_COLS],
    'coupling': [c for c in w.columns if c.startswith('coupling_')],
    'motion':   [c for c in w.columns if c.startswith('motion_')],
}
N_FEATURE_COLS = sum(len(v) for v in FAMILIES.values())
print(f"\n{len(PARTICIPANTS)} participants, {N_FEATURE_COLS} behavioural feature columns "
      f"across {len(FAMILIES)} families")


Loaded ../../data/processed/windows.parquet
2,684 windows x 596 columns

17 participants, 574 behavioural feature columns across 5 families


In [3]:
df = w
df

,sessionId,participantId,deviceFamily,window_index,window_start_s,window_end_s,taskIndex,taskType,activeArea,task_pass,task_start_s,task_end_s,typing_afforded,tapping_afforded,scrolling_afforded,typing_observed,tapping_observed,scrolling_observed,task_boundary_straddle,typing_straddle_conflict,tapping_straddle_conflict,scrolling_straddle_conflict,typing_keydown_count,typing_backspace_share,typing_autorepeat_share,typing_share_letter,typing_share_space,typing_share_backspace,typing_share_digit,typing_share_punct_or_symbol,...,motion_idle_run_max_len_s,motion_idle_sway_power,motion_idle_tremor_power,motion_idle_accel_mag_mean,motion_idle_accel_mag_std,motion_idle_accel_mag_median,motion_idle_accel_mag_iqr,motion_idle_accel_mag_p95,motion_idle_accel_mag_max,motion_idle_accel_mag_n,motion_idle_accel_mag_cv,motion_idle_accel_mag_slope,motion_active_accel_mag_mean,motion_active_accel_mag_std,motion_active_accel_mag_median,motion_active_accel_mag_iqr,motion_active_accel_mag_p95,motion_active_accel_mag_max,motion_active_accel_mag_n,motion_active_accel_mag_cv,motion_active_accel_mag_slope,motion_cross_axis_corr_mean,motion_cross_axis_corr_std,motion_cross_axis_corr_median,motion_cross_axis_corr_iqr,motion_cross_axis_corr_p95,motion_cross_axis_corr_max,motion_cross_axis_corr_n,motion_cross_axis_corr_cv,motion_cross_axis_corr_slope
0,04c75fb82aca415db10da08431699afc,pEAB9GS,mobile,0,0.001,15.001,NaN,None,None,NaN,NaN,NaN,<NA>,<NA>,<NA>,False,True,True,False,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.20,NaN,NaN,0.943619,0.539846,0.903702,0.715753,1.869589,2.574084,44.0,0.572101,-0.473420,0.379458,0.314167,0.295288,0.331204,0.955498,1.997823,224.0,0.827937,-0.023576,-0.196720,0.422252,-0.150393,0.840539,0.391324,0.421914,231.0,2.146462,-0.077261
1,04c75fb82aca415db10da08431699afc,pEAB9GS,mobile,1,7.501,22.501,1.0,tap_account,home,1.0,2.932,7.688,False,True,True,False,True,True,True,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.25,NaN,NaN,0.283899,0.131527,0.305836,0.187533,0.411030,0.662913,25.0,0.463286,0.125351,0.386837,0.514059,0.240932,0.335125,0.930812,3.732556,218.0,1.328878,0.024969,-0.534261,0.270052,-0.597827,0.156888,0.103548,0.421914,238.0,0.505468,-0.025093
2,04c75fb82aca415db10da08431699afc,pEAB9GS,mobile,2,15.001,30.001,2.0,home_explore,home,1.0,7.688,15.282,False,True,True,True,True,True,True,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.25,NaN,NaN,0.283899,0.131527,0.305836,0.187533,0.411030,0.662913,25.0,0.463286,0.125351,0.598681,0.601501,0.419834,0.523975,1.711720,3.732556,212.0,1.004710,0.006080,-0.709677,0.129685,-0.739869,0.232101,-0.549524,-0.403638,235.0,0.182739,-0.021475
3,04c75fb82aca415db10da08431699afc,pEAB9GS,mobile,3,22.501,37.501,3.0,typing_search,activity,1.0,15.282,36.379,True,True,True,True,True,True,True,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.10,NaN,NaN,0.105814,0.059660,0.105814,0.059660,0.159508,0.165474,2.0,0.563814,NaN,0.494169,0.431488,0.371611,0.427490,1.406838,2.289117,198.0,0.873160,-0.044054,-0.659440,0.273348,-0.755419,0.213396,0.061671,0.273352,198.0,0.414516,0.046517
4,04c75fb82aca415db10da08431699afc,pEAB9GS,mobile,4,30.001,45.001,3.0,typing_search,activity,1.0,15.282,36.379,True,True,True,False,True,True,True,True,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.55,NaN,NaN,0.103796,0.090618,0.069079,0.100285,0.217130,0.460397,45.0,0.873035,0.011886,0.296588,0.310418,0.202047,0.241293,0.997558,1.823113,150.0,1.046631,0.006239,-0.507017,0.370412,-0.617775,0.590742,0.133106,0.273352,192.0,0.730572,-0.019561
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2679,f7309697facd4ea6885b5b3f3d5e744f,pUNKH7L,mobile,18,135.055,150.055,16.0,transaction_feed,activity,2.0,122.560,136.990,False,True,True,False,True,True,True,True,False,False,NaN,NaN,NaN,NaN,NaN,Na

In [ ]:
family_columns = df.columns.str.contains('motion')
df.loc[:,family_columns]

,motion_idle_frac,motion_idle_run_max_len_s,motion_idle_sway_power,motion_idle_tremor_power,motion_idle_accel_mag_mean,motion_idle_accel_mag_std,motion_idle_accel_mag_median,motion_idle_accel_mag_iqr,motion_idle_accel_mag_p95,motion_idle_accel_mag_max,motion_idle_accel_mag_n,motion_idle_accel_mag_cv,motion_idle_accel_mag_slope,motion_active_accel_mag_mean,motion_active_accel_mag_std,motion_active_accel_mag_median,motion_active_accel_mag_iqr,motion_active_accel_mag_p95,motion_active_accel_mag_max,motion_active_accel_mag_n,motion_active_accel_mag_cv,motion_active_accel_mag_slope,motion_cross_axis_corr_mean,motion_cross_axis_corr_std,motion_cross_axis_corr_median,motion_cross_axis_corr_iqr,motion_cross_axis_corr_p95,motion_cross_axis_corr_max,motion_cross_axis_corr_n,motion_cross_axis_corr_cv,motion_cross_axis_corr_slope
0,0.164179,2.20,NaN,NaN,0.943619,0.539846,0.903702,0.715753,1.869589,2.574084,44.0,0.572101,-0.473420,0.379458,0.314167,0.295288,0.331204,0.955498,1.997823,224.0,0.827937,-0.023576,-0.196720,0.422252,-0.150393,0.840539,0.391324,0.421914,231.0,2.146462,-0.077261
1,0.102881,1.25,NaN,NaN,0.283899,0.131527,0.305836,0.187533,0.411030,0.662913,25.0,0.463286,0.125351,0.386837,0.514059,0.240932,0.335125,0.930812,3.732556,218.0,1.328878,0.024969,-0.534261,0.270052,-0.597827,0.156888,0.103548,0.421914,238.0,0.505468,-0.025093
2,0.105485,1.25,NaN,NaN,0.283899,0.131527,0.305836,0.187533,0.411030,0.662913,25.0,0.463286,0.125351,0.598681,0.601501,0.419834,0.523975,1.711720,3.732556,212.0,1.004710,0.006080,-0.709677,0.129685,-0.739869,0.232101,-0.549524,-0.403638,235.0,0.182739,-0.021475
3,0.010000,0.10,NaN,NaN,0.105814,0.059660,0.105814,0.059660,0.159508,0.165474,2.0,0.563814,NaN,0.494169,0.431488,0.371611,0.427490,1.406838,2.289117,198.0,0.873160,-0.044054,-0.659440,0.273348,-0.755419,0.213396,0.061671,0.273352,198.0,0.414516,0.046517
4,0.230769,1.55,NaN,NaN,0.103796,0.090618,0.069079,0.100285,0.217130,0.460397,45.0,0.873035,0.011886,0.296588,0.310418,0.202047,0.241293,0.997558,1.823113,150.0,1.046631,0.006239,-0.507017,0.370412,-0.617775,0.590742,0.133106,0.273352,192.0,0.730572,-0.019561
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2679,0.000000,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.273287,0.275821,0.173381,0.210915,0.889421,1.467875,193.0,1.009274,-0.010903,0.301444,0.510895,0.369089,0.463877,0.847021,0.856252,186.0,1.694825,-0.089997
2680,0.033457,0.45,NaN,NaN,0.105701,0.022404,0.104361,0.024144,0.141328,0.159685,9.0,0.211954,-0.038095,0.280284,0.319492,0.160000,0.254935,0.919726,2.732609,260.0,1.139884,-0.007783,-0.380074,0.598671,-0.736218,1.166973,0.543309,0.856252,260.0,1.575144,-0.124844
2681,0.038136,0.45,NaN,NaN,0.105701,0.022404,0.104361,0.024144,0.141328,0.159685,9.0,0.211954,-0.038095,0.303677,0.326878,0.179519,0.285173,0.914815,2.732609,227.0,1.076399,0.000301,-0.866036,0.109746,-0.875264,0.211983,-0.692678,-0.363827,229.0,0.126722,-0.006107
2682,0.000000,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.257759,0.226014,0.173714,0.227041,0.767856,1.260079,169.0,0.876841,-0.005052,-0.756987,0.290784,-0.854793,0.110714,0.027924,0.149971,162.0,0.384133,0.055931


In [24]:
df.isna().sum().sort_values(ascending=False).head(50)

typing_transition_space_space_ms_std           2684
typing_transition_space_backspace_ms_slope     2684
typing_transition_backspace_space_ms_n         2684
typing_transition_backspace_space_ms_max       2684
typing_transition_backspace_space_ms_p95       2684
typing_transition_backspace_space_ms_iqr       2684
typing_transition_backspace_space_ms_median    2684
typing_transition_backspace_space_ms_std       2684
typing_transition_backspace_space_ms_mean      2684
typing_transition_backspace_letter_ms_slope    2684
typing_transition_space_backspace_ms_cv        2684
typing_transition_backspace_space_ms_slope     2684
typing_transition_space_backspace_ms_p95       2684
typing_transition_letter_backspace_ms_slope    2684
typing_transition_space_backspace_ms_iqr       2684
typing_transition_space_backspace_ms_std       2684
typing_transition_space_space_ms_iqr           2684
typing_transition_space_space_ms_p95           2684
typing_transition_space_space_ms_cv            2684
typing_dwell